# Moduł 3: Pydantic - Walidacja i Serializacja Danych

**Ćwiczenie:** #3 - Pydantic Schemas

---

## Spis treści

1. [Wprowadzenie](#1-wprowadzenie)
2. [Kluczowe koncepcje](#2-kluczowe-koncepcje)
   - [BaseModel - Podstawowa klasa](#21-basemodel---podstawowa-klasa)
   - [Opcjonalne pola i wartości domyślne](#22-opcjonalne-pola-i-wartości-domyślne)
   - [Field validators](#23-field-validators---zaawansowana-walidacja)
   - [Nested Models](#24-nested-models-zagnieżdżone-modele)
   - [Listy i kolekcje](#25-listy-i-kolekcje)
   - [Użycie w FastAPI](#26-użycie-w-fastapi-request-body)
   - [A co z walidacją query params?](#27-a-co-z-walidacją-query-params)
   - [Response Model](#28-response-model-wymuszenie-struktury-odpowiedzi)
   - [Model Config](#29-model-config-zaawansowane-opcje)
3. [Best Practices](#3-best-practices)
4. [Demo Live Coding](#4-demo-live-coding)
5. [Przygotowanie do ćwiczenia](#5-przygotowanie-do-ćwiczenia)
6. [Dodatkowe materiały](#6-dodatkowe-materiały)

---

## Materiały zależne

Zanim zaczniesz ten moduł, upewnij się że znasz:
- **Type annotations** - zobacz `materials/type_annotations.ipynb`
- **Pydantic intro** - zobacz `materials/pydantic_intro.ipynb`

## 1. Wprowadzenie

### Po co ten temat?

**Pydantic** to biblioteka do:
- **Walidacji danych** - automatyczne sprawdzanie typów i wartości
- **Serializacji** - konwersja Python objects ↔ JSON
- **Dokumentacji** - automatyczne generowanie schema w Swagger

FastAPI używa Pydantic jako fundamentu do **data validation** i **serialization**.

### Gdzie to użyjemy w praktyce?

- **Request body validation** - POST/PUT/PATCH requests
- **Response schemas** - wymuszenie struktury odpowiedzi
- **Query parameters** - złożone filtry i validation
- **Configuration** - walidacja env variables i config files

**Dlaczego Pydantic?**
- Wykorzystuje **type hints** (native Python 3.6+)
- **Szybki** - napisany w Rust (v2)
- **Automatyczna dokumentacja** - OpenAPI schema generation
- **Bardzo dokładne error messages** - pokazuje dokładnie co jest nie tak

## 2. Kluczowe koncepcje

### 2.1. BaseModel - Podstawowa klasa

**Najprostszy przykład:**

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    email: str
    age: int

# Tworzenie instancji
user = User(id=1, name="John", email="john@example.com", age=30)

print(user.id)     # → 1
print(user.name)   # → John
print(user.model_dump())  # → {'id': 1, 'name': 'John', ...}
print(user.model_dump_json())  # → JSON string

**Automatyczna walidacja:**

In [ ]:
# ✅ OK
user = User(id=1, name="John", email="john@example.com", age=30)

# ❌ ValidationError - age nie jest int
# user = User(id=1, name="John", email="john@example.com", age="thirty")
# → ValidationError: value is not a valid integer

# ❌ ValidationError - brak wymaganego pola
# user = User(id=1, name="John")
# → ValidationError: field required

### 2.2. Opcjonalne pola i wartości domyślne

#### Default values:

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    email: str
    age: int = 18          # Domyślna wartość
    is_active: bool = True  # Domyślna wartość

# Możesz pominąć pola z defaults
user = User(id=1, name="John", email="john@example.com")
print(user.age)        # → 18 (default)
print(user.is_active)  # → True (default)

#### Optional fields (None):

In [ ]:
from typing import Optional

class User(BaseModel):
    id: int
    name: str
    email: str
    bio: Optional[str] = None  # Może być None

# Bio może być pominięte
user = User(id=1, name="John", email="john@example.com")
print(user.bio)  # → None

# Lub podane
user = User(id=1, name="John", email="john@example.com", bio="Python dev")
print(user.bio)  # → "Python dev"

**Różnica:**
- **Default value** (`age: int = 18`) - pole opcjonalne, ale musi być int
- **Optional** (`bio: Optional[str] = None`) - pole opcjonalne, może być None

### 2.3. Field validators - Zaawansowana walidacja

#### Built-in validators:

In [ ]:
from pydantic import BaseModel, Field

class User(BaseModel):
    id: int = Field(gt=0)  # Greater than 0
    name: str = Field(min_length=2, max_length=50)
    email: str = Field(pattern=r'^[\w\.-]+@[\w\.-]+\.\w+$')
    age: int = Field(ge=18, le=120)  # Greater or equal, Less or equal

# ✅ OK
user = User(id=1, name="John", email="john@example.com", age=30)

# ❌ ValidationError - id musi być > 0
# user = User(id=0, name="John", email="john@example.com", age=30)

# ❌ ValidationError - name zbyt krótki
# user = User(id=1, name="J", email="john@example.com", age=30)

# ❌ ValidationError - age < 18
# user = User(id=1, name="John", email="john@example.com", age=15)

**Dostępne validators:**
- **Liczby**: `gt`, `ge`, `lt`, `le`, `multiple_of`
- **Stringi**: `min_length`, `max_length`, `pattern` (regex)
- **Listy**: `min_items`, `max_items`
- **Wszystkie**: `description` (dokumentacja)

#### Custom validators:

In [ ]:
from pydantic import BaseModel, field_validator

class User(BaseModel):
    name: str
    email: str
    password: str

    @field_validator('email')
    @classmethod
    def email_must_be_valid(cls, v):
        """Custom email validation"""
        if '@' not in v:
            raise ValueError('must contain @')
        if not v.endswith('.com') and not v.endswith('.org'):
            raise ValueError('must end with .com or .org')
        return v

    @field_validator('password')
    @classmethod
    def password_strength(cls, v):
        """Password musi mieć min. 8 znaków i cyfrę"""
        if len(v) < 8:
            raise ValueError('must be at least 8 characters')
        if not any(char.isdigit() for char in v):
            raise ValueError('must contain at least one digit')
        return v

# ✅ OK
user = User(name="John", email="john@example.com", password="pass1234")

# ❌ ValidationError - email bez @
# user = User(name="John", email="john.example.com", password="pass1234")

# ❌ ValidationError - password zbyt krótki
# user = User(name="John", email="john@example.com", password="pass")

# ❌ ValidationError - password bez cyfry
# user = User(name="John", email="john@example.com", password="password")

### 2.4. Nested Models (zagnieżdżone modele)

**Przykład: Address w User**

In [ ]:
from pydantic import BaseModel
from typing import Optional

class Address(BaseModel):
    street: str
    city: str
    country: str
    zip_code: Optional[str] = None

class User(BaseModel):
    id: int
    name: str
    email: str
    address: Address  # Zagnieżdżony model!

# Tworzenie
user = User(
    id=1,
    name="John",
    email="john@example.com",
    address={
        "street": "Main St 123",
        "city": "Warsaw",
        "country": "Poland"
    }
)

print(user.address.city)  # → Warsaw
print(user.model_dump())

**Opcjonalny nested model:**

In [ ]:
class User(BaseModel):
    id: int
    name: str
    address: Optional[Address] = None  # Opcjonalny

user = User(id=1, name="John")
print(user.address)  # → None

### 2.5. Listy i kolekcje

**Lista prostych typów:**

In [ ]:
from pydantic import BaseModel
from typing import List

class TaskList(BaseModel):
    name: str
    tags: List[str]  # Lista stringów
    priorities: List[int]  # Lista intów

tasks = TaskList(
    name="My Tasks",
    tags=["work", "urgent"],
    priorities=[1, 2, 3]
)

print(tasks.tags)  # → ['work', 'urgent']

**Lista zagnieżdżonych modeli:**

In [ ]:
class Post(BaseModel):
    id: int
    title: str
    content: str

class Blog(BaseModel):
    name: str
    posts: List[Post]  # Lista modeli!

blog = Blog(
    name="My Blog",
    posts=[
        {"id": 1, "title": "First Post", "content": "Hello"},
        {"id": 2, "title": "Second Post", "content": "World"}
    ]
)

print(blog.posts[0].title)  # → First Post

### 2.6. Użycie w FastAPI (Request Body)

**Podstawowy przykład:**

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI()

class UserCreate(BaseModel):
    name: str = Field(min_length=2, max_length=50)
    email: str
    password: str = Field(min_length=8)
    age: int = Field(ge=18, le=120)

@app.post("/users")
def create_user(user: UserCreate):  # Walidajca
    """
    Stwórz użytkownika

    FastAPI automatycznie:
    1. Parsuje JSON z body (nie query params!)
    2. Waliduje według UserCreate schema
    3. Zwraca 422 jeśli validation error
    """
    return {
        "message": "User created",
        "user": user.model_dump()  # serializacja
    }

**Request:**
```bash
# ✅ OK
curl -X POST http://localhost:8000/users \
  -H "Content-Type: application/json" \
  -d '{"name": "John", "email": "john@example.com", "password": "pass1234", "age": 30}'

# → 200 OK

# ❌ Validation Error - password zbyt krótki
curl -X POST http://localhost:8000/users \
  -H "Content-Type: application/json" \
  -d '{"name": "John", "email": "john@example.com", "password": "pass", "age": 30}'

# → 422 Unprocessable Entity
```

**FastAPI automatycznie:**
- Parsuje JSON → Pydantic model
- Waliduje wszystkie pola
- Zwraca **422 Unprocessable Entity** z dokładnymi error details
- Generuje schema w Swagger UI

### 2.7. A co z walidacją query params?

**Problem:** Widziałeś jak Pydantic waliduje request body. A co z query params?

**Proste query params (bez walidacji):**

In [ ]:
@app.get("/users")
def get_users(name: str | None = None, age: int | None = None):
    """
    Query params: /users?name=John&age=30
    
    Problem:
    - Brak zaawansowanej walidacji (min/max age, pattern dla name)
    - Każdy param osobno w sygnaturze (brzydko przy wielu params)
    - Brak dokumentacji constraints
    """
    return {"name": name, "age": age}

**Rozwiązanie: Pydantic + Depends()**

**KLUCZOWA RÓŻNICA:**
- `user: UserCreate` → parsuje **BODY** (JSON)
- `filters: UserFilters = Depends()` → parsuje **QUERY PARAMS**

In [ ]:
from fastapi import Depends
from pydantic import BaseModel, Field
from typing import Optional

class UserFilters(BaseModel):
    """Schema dla query params"""
    name: Optional[str] = Field(None, min_length=2, max_length=50)
    age: Optional[int] = Field(None, ge=18, le=120)
    limit: int = Field(10, ge=1, le=100, description="Results per page")

@app.get("/users")
def get_users(filters: UserFilters = Depends()):
    """
    Query params: /users?name=John&age=30&limit=20
    
    FastAPI automatycznie:
    1. Parsuje query params → UserFilters
    2. Waliduje (age >= 18, limit 1-100, etc.)
    3. Zwraca 422 jeśli validation error
    """
    return {
        "filters": filters.model_dump(),
        "message": f"Searching users: {filters.name}, age {filters.age}"
    }

# ✅ OK: /users?name=John&age=30
# ✅ OK: /users?limit=50 (name i age opcjonalne)
# ❌ 422: /users?age=15 (age < 18)
# ❌ 422: /users?limit=200 (limit > 100)

**Podsumowanie:**

| Gdzie | Jak | Przykład |
|-------|-----|----------|
| **Body (JSON)** | `user: UserCreate` | POST /users |
| **Query params** | `filters: UserFilters = Depends()` | GET /users?name=... |
| **Path params** | `user_id: int` | GET /users/{user_id} |

**Więcej o Depends() w module #5 (Dependency Injection)**

### 2.8. Response Model (wymuszenie struktury odpowiedzi)

**Dlaczego response_model?**
- Dokumentacja (Swagger pokazuje dokładny format)
- Filtrowanie pól (np. ukrycie password)
- Walidacja response (wykrycie błędów w twoim kodzie)

**Przykład:**

In [ ]:
from pydantic import BaseModel

class UserCreate(BaseModel):
    """Schema dla tworzenia użytkownika"""
    name: str
    email: str
    password: str

class UserResponse(BaseModel):
    """Schema dla odpowiedzi (BEZ password!)"""
    id: int
    name: str
    email: str

@app.post("/users", response_model=UserResponse)
def create_user(user: UserCreate):
    """
    Zwróć UserResponse (bez password)
    """
    # W rzeczywistości: zapis do DB, hashowanie password
    # Tu: mock
    return {
        "id": 123,
        "name": user.name,
        "email": user.email,
        "password": user.password  # To zostanie ZIGNOROWANE!
    }

**Response:**
```json
{
  "id": 123,
  "name": "John",
  "email": "john@example.com"
}
```

**Zauważ:** Mimo że zwracamy `password`, FastAPI go **filtruje** bo nie ma go w `UserResponse`!

### 2.9. Model Config (zaawansowane opcje)

In [ ]:
from pydantic import BaseModel, ConfigDict

class User(BaseModel):
    model_config = ConfigDict(
        # Zezwól na dodatkowe pola (nie w schema)
        extra='allow',

        # Sprawdź assignments po utworzeniu
        validate_assignment=True,

        # Użyj aliasów dla pól
        populate_by_name=True,
    )

    id: int
    name: str

**Przydatne opcje:**
- `extra='forbid'` - błąd jeśli dodatkowe pola w JSON
- `extra='allow'` - zezwól na dodatkowe pola (domyślnie ignorowane)
- `validate_assignment=True` - waliduj też przy `user.name = "new"`

## 3. Best Practices

### ✅ Rób tak:

**1. Rozdziel input i output schemas:**

In [ ]:
# ✅ Dobre - osobne schema dla create i response
class UserCreate(BaseModel):
    """Request body - Z password"""
    name: str
    email: str
    password: str

class UserResponse(BaseModel):
    """Response - BEZ password"""
    id: int
    name: str
    email: str

@app.post("/users", response_model=UserResponse)
def create_user(user: UserCreate):
    # Automatycznie filtruje password z odpowiedzi
    return {...}

**2. Używaj Field() dla dokumentacji i walidacji:**

In [ ]:
# ✅ Dobre
from pydantic import Field

class User(BaseModel):
    name: str = Field(
        min_length=2,
        max_length=50,
        description="User's full name"
    )
    age: int = Field(
        ge=18,
        le=120,
        description="User's age (must be 18+)"
    )

**3. Dodawaj examples dla Swagger:**

In [ ]:
# ✅ Dobre
class User(BaseModel):
    model_config = ConfigDict(
        json_schema_extra={
            "examples": [
                {
                    "name": "John Doe",
                    "email": "john@example.com",
                    "age": 30
                }
            ]
        }
    )

    name: str
    email: str
    age: int

**4. Używaj Enum dla ograniczonych wartości:**

In [ ]:
# ✅ Dobre
from enum import Enum

class UserRole(str, Enum):
    admin = "admin"
    user = "user"
    guest = "guest"

class User(BaseModel):
    name: str
    role: UserRole  # Tylko admin/user/guest

**5. Grupuj powiązane fields w nested models:**

In [ ]:
# ✅ Dobre
class Address(BaseModel):
    street: str
    city: str
    country: str

class User(BaseModel):
    name: str
    address: Address  # Zagnieżdżony model

# ❌ Złe - wszystko na płasko
# class User(BaseModel):
#     name: str
#     street: str
#     city: str
#     country: str

### ❌ Nie rób tak:

**1. Nie pomijaj type hints**

**2. Nie używaj dict zamiast Pydantic model**

**3. Nie zwracaj password w response**

**4. Nie dubluj validation logic**

**5. Nie ignoruj validation errors**